# TALOS — Embed Pending (Colab)
Lee `pending_<ts>.jsonl` de Drive, computa embeddings e5 (384-dim, L2, prefijo `passage: `) y escribe `vectors_<ts>.jsonl`.

**In-situ**: valida el embedding contract embebido; si difiere del local, se aborta (riesgo de mezclar espacios vectoriales).


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive montado")

In [ ]:
import os, json, glob

# EDITAR si la carpeta difiere
DRIVE_FOLDER = "/content/drive/MyDrive/talos_reindex"
MODEL_ONNX   = f"{DRIVE_FOLDER}/model.onnx"
TOKENIZER    = f"{DRIVE_FOLDER}/tokenizer.json"

pendings = sorted(glob.glob(f"{DRIVE_FOLDER}/pending_*.jsonl"))
assert pendings, "No hay pending_*.jsonl en Drive. Correr primero: talos export-pending y subir el archivo."
PENDING_PATH = pendings[-1]
print("Pendiente:", PENDING_PATH)

with open(PENDING_PATH) as f:
    first = json.loads(f.readline())
CONTRACT = first["_meta"]["contract"]
EXPECTED = first["_meta"]["count"]
print("Contract:", json.dumps(CONTRACT, indent=2)[:400])
print("Chunks a embedder:", EXPECTED)

In [ ]:
!pip install -q --upgrade --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/ onnxruntime-gpu tokenizers numpy tqdm

In [ ]:
import onnxruntime as ort
import numpy as np
from tokenizers import Tokenizer

available = ort.get_available_providers()
PROVIDER = "CUDAExecutionProvider" if "CUDAExecutionProvider" in available else "CPUExecutionProvider"
print("Providers:", available, "->", PROVIDER)

tokenizer = Tokenizer.from_file(TOKENIZER)
tokenizer.enable_truncation(max_length=CONTRACT["max_length"])
tokenizer.enable_padding(pad_id=0, pad_token="<pad>", length=None)

opts = ort.SessionOptions()
opts.intra_op_num_threads = 4
session = ort.InferenceSession(MODEL_ONNX, sess_options=opts, providers=[PROVIDER])

def embed_passages(texts):
    formatted = [f'{CONTRACT["prefixes"]["passage"]}{t.strip()}' for t in texts]
    enc = tokenizer.encode_batch(formatted)
    ids  = np.array([e.ids for e in enc], dtype=np.int64)
    mask = np.array([e.attention_mask for e in enc], dtype=np.int64)
    tids = np.array([e.type_ids for e in enc], dtype=np.int64)
    out = session.run(None, {"input_ids": ids, "attention_mask": mask, "token_type_ids": tids})
    lhs = out[0]
    m = np.expand_dims(mask, -1).astype(np.float32)
    emb = np.sum(lhs * m, axis=1) / np.clip(m.sum(axis=1), 1e-9, None)
    if CONTRACT["normalize_l2"]:
        nrm = np.linalg.norm(emb, axis=1, keepdims=True)
        emb = emb / np.clip(nrm, 1e-9, None)
    return emb.tolist()

probe = embed_passages(["test de humo"])
assert len(probe[0]) == CONTRACT["dimensions"]
print("Smoke test OK — dim", len(probe[0]))

In [ ]:
import time
from tqdm.notebook import tqdm

records = []
with open(PENDING_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            obj = json.loads(line)
            if "_meta" not in obj:
                records.append(obj)
assert len(records) == EXPECTED, f"{len(records)} != {EXPECTED}"

ts = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
OUT_PATH = f"{DRIVE_FOLDER}/vectors_{ts}.jsonl"
BATCH = 128

t0 = time.time()
with open(OUT_PATH, "w") as out:
    out.write(json.dumps({"_meta": {"schema": "talos-vectors-v1", "contract": CONTRACT,
                                    "created_utc": ts, "count": len(records)}}, ensure_ascii=False) + "\n")
    for i in tqdm(range(0, len(records), BATCH), desc="Embedding"):
        batch = records[i:i+BATCH]
        vecs = embed_passages([r["text"] for r in batch])
        for r, v in zip(batch, vecs):
            out.write(json.dumps({"chunk_id": r["chunk_id"], "notebook_id": r["notebook_id"],
                                  "vector": v}) + "\n")

print(f"Escrito {OUT_PATH} ({len(records)} vectores en {(time.time()-t0):.0f}s)")
print("Ahora en local: talos sync-vectors <archivo.jsonl>")

## Vuelta local
1. El archivo `vectors_<ts>.jsonl` queda en Drive.
2. En el repo: `talos sync-vectors data/colab/vectors_<ts>.jsonl`
3. `talos verify-index` para confirmar 0 pendientes + integridad.
